# MediPredict: Diabetes Prediction Model
## Google Colab Notebook

This notebook sets up and runs the MediPredict diabetes prediction model in Google Colab with an interactive interface for making predictions.

In [ ]:
# 1. Install and Import Required Libraries
import subprocess
import sys

# Install required packages
packages = ['tensorflow>=2.16.1', 'numpy', 'pandas', 'scikit-learn', 'gradio']
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All packages installed successfully!")

# Import libraries
import numpy as np
import pandas as pd
import json
import os
from tensorflow import keras

print("✓ Libraries imported successfully!")

In [ ]:
# 2. Mount Google Drive (to access your MediPredict model and data)
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully!")
print("Your Drive files are available at /content/drive/My Drive/")

In [ ]:
# 3. Download or Load MediPredict Model
# Download the model and supporting files from the repository
import urllib.request
import zipfile
import os

model_url = "https://github.com/your-repo/MediPredict/raw/main/MediPredict/models/"
os.makedirs('/content/models', exist_ok=True)

# Files to download
files_to_download = [
    ('diabetes_model.keras', 'diabetes_model.keras'),
    ('imputation_medians.json', 'imputation_medians.json'),
    ('metrics.json', 'metrics.json')
]

for filename, local_name in files_to_download:
    try:
        url = model_url + filename
        filepath = f'/content/models/{local_name}'
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"✓ {filename} downloaded")
    except Exception as e:
        print(f"⚠ Could not download {filename}: {e}")
        print(f"  If this fails, upload the file manually to /content/models/")

print("\nTo use local files: Upload the models folder from your MediPredict project to Drive, then use:")
print("  tf_model = keras.models.load_model('/content/drive/My Drive/models/diabetes_model.keras')")

In [ ]:
# 4. Load the Model and Supporting Files
import json

# Try to load from local/downloaded files first
model_path = '/content/models/diabetes_model.keras'
medians_path = '/content/models/imputation_medians.json'
metrics_path = '/content/models/metrics.json'

try:
    # Load TensorFlow model
    tf_model = keras.models.load_model(model_path)
    print("✓ Model loaded successfully!")
    
    # Load imputation medians
    with open(medians_path, 'r') as f:
        imputation_medians = json.load(f)
    print("✓ Imputation medians loaded")
    
    # Load metrics
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    print("✓ Model metrics loaded")
    
    print(f"\nModel Metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value:.4f}")
        
except Exception as e:
    print(f"⚠ Could not load model from /content/models/: {e}")
    print("\nTo use your model:")
    print("  1. Upload the MediPredict/models folder to your Google Drive")
    print("  2. Update the paths below to point to your Drive files")
    print("  3. Re-run this cell")
    
    # Create dummy values for demonstration
    imputation_medians = {'Glucose': 117.0, 'BloodPressure': 76.0, 'SkinThickness': 29.0, 'Insulin': 79.5, 'BMI': 32.0}
    metrics = {'accuracy': 0.7922, 'precision': 0.7347, 'recall': 0.6545, 'f1_score': 0.6923}

In [ ]:
# 5. Define Prediction Function
def predict_diabetes(pregnancies, glucose, blood_pressure, skin_thickness, insulin, bmi, dpf, age):
    """
    Predict diabetes risk using the MediPredict model
    
    Parameters:
    - pregnancies: Number of pregnancies (0-25)
    - glucose: Glucose level (10-500 mg/dL)
    - blood_pressure: Blood pressure (10-300 mmHg)
    - skin_thickness: Skin thickness (0-120 mm)
    - insulin: Insulin level (0-2000 uIU/mL)
    - bmi: Body Mass Index (5-100)
    - dpf: Diabetes Pedigree Function (0-5)
    - age: Age (1-120 years)
    
    Returns:
    - Dictionary with prediction, confidence, and recommendations
    """
    
    # Validate inputs
    features = {
        'Pregnancies': pregnancies,
        'Glucose': glucose,
        'BloodPressure': blood_pressure,
        'SkinThickness': skin_thickness,
        'Insulin': insulin,
        'BMI': bmi,
        'DiabetesPedigreeFunction': dpf,
        'Age': age
    }
    
    # Impute missing values
    for key, median_val in imputation_medians.items():
        if features.get(key, 0) == 0:
            features[key] = median_val
    
    # Prepare input for model
    input_data = np.array([[
        features['Pregnancies'],
        features['Glucose'],
        features['BloodPressure'],
        features['SkinThickness'],
        features['Insulin'],
        features['BMI'],
        features['DiabetesPedigreeFunction'],
        features['Age']
    ]], dtype=np.float32)
    
    # Make prediction
    prediction_prob = tf_model.predict(input_data, verbose=0)[0][0]
    prediction = 1 if prediction_prob >= 0.5 else 0
    confidence = float(prediction_prob) if prediction == 1 else float(1 - prediction_prob)
    
    # Generate recommendations
    recommendations = []
    if prediction == 1:
        recommendations.append("Consult a healthcare professional for a diagnostic glucose tolerance test (OGTT) or HbA1c screening.")
        if glucose > 140:
            recommendations.append("Your glucose level is elevated. Consider monitoring your carbohydrate intake.")
        if bmi > 25:
            recommendations.append("Adopting a low-glycemic, calorie-controlled diet and regular exercise can help improve insulin sensitivity.")
    else:
        recommendations.append("Maintain your healthy routine! Focus on a balanced diet rich in fiber, whole grains, and lean proteins.")
        if glucose > 100:
            recommendations.append("Your fasting glucose is in the pre-diabetic range (100-125 mg/dL). Consider reducing refined sugars.")
    
    if blood_pressure > 80:
        recommendations.append("Your blood pressure is elevated. Reducing sodium intake is recommended.")
    
    if age > 45:
        recommendations.append("Annual metabolic screenings are recommended for individuals over 45 years of age.")
    
    label = "Likely Diabetic" if prediction == 1 else "Likely Non-Diabetic"
    
    return {
        'prediction': int(prediction),
        'label': label,
        'confidence': confidence,
        'recommendations': recommendations,
        'input_features': features
    }

print("✓ Prediction function defined!")

In [ ]:
# 6. Create Interactive Prediction Interface with Gradio
import gradio as gr

def gradio_predict(pregnancies, glucose, blood_pressure, skin_thickness, insulin, bmi, dpf, age):
    """Wrapper for Gradio interface"""
    result = predict_diabetes(pregnancies, glucose, blood_pressure, skin_thickness, insulin, bmi, dpf, age)
    
    # Format output
    output_text = f"""
    **Prediction Result: {result['label']}**
    
    Confidence: {result['confidence']:.2%}
    
    **Health Recommendations:**
    """
    
    for i, rec in enumerate(result['recommendations'], 1):
        output_text += f"\n{i}. {rec}"
    
    return output_text

# Create Gradio interface
interface = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Slider(0, 25, step=1, label="Number of Pregnancies", value=0),
        gr.Slider(10, 500, step=1, label="Glucose Level (mg/dL)", value=120),
        gr.Slider(10, 300, step=1, label="Blood Pressure (mmHg)", value=80),
        gr.Slider(0, 120, step=1, label="Skin Thickness (mm)", value=20),
        gr.Slider(0, 2000, step=10, label="Insulin Level (uIU/mL)", value=80),
        gr.Slider(5, 100, step=0.1, label="BMI", value=25),
        gr.Slider(0, 5, step=0.01, label="Diabetes Pedigree Function", value=0.5),
        gr.Slider(1, 120, step=1, label="Age (years)", value=30)
    ],
    outputs=gr.Textbox(label="Prediction & Recommendations", lines=10),
    title="MediPredict: Diabetes Risk Assessment",
    description="Enter your health metrics to get a diabetes risk prediction with personalized recommendations.",
    examples=[
        [0, 148, 60, 29, 170, 38.5, 0.423, 43],  # Example of diabetic
        [2, 100, 60, 0, 0, 30.7, 0.179, 28],     # Example of non-diabetic
    ]
)

# Launch the interface
interface.launch(share=True)

In [ ]:
# 7. Test Predictions with Sample Data
print("=" * 60)
print("TESTING PREDICTIONS WITH SAMPLE DATA")
print("=" * 60)

# Sample data - likely diabetic
print("\n✓ Sample 1: High-risk individual (likely diabetic)")
result1 = predict_diabetes(
    pregnancies=6, glucose=148, blood_pressure=72, skin_thickness=35,
    insulin=155, bmi=33.6, dpf=0.627, age=50
)
print(f"  Prediction: {result1['label']}")
print(f"  Confidence: {result1['confidence']:.2%}")

# Sample data - likely non-diabetic
print("\n✓ Sample 2: Low-risk individual (likely non-diabetic)")
result2 = predict_diabetes(
    pregnancies=1, glucose=107, blood_pressure=60, skin_thickness=25,
    insulin=86, bmi=26.4, dpf=0.133, age=23
)
print(f"  Prediction: {result2['label']}")
print(f"  Confidence: {result2['confidence']:.2%}")

print("\n" + "=" * 60)
print("Predictions completed successfully!")

In [ ]:
# 8. Save Results to Google Drive
import json
import os
from datetime import datetime

# Create output directory in Drive
output_dir = '/content/drive/My Drive/MediPredict_Results'
os.makedirs(output_dir, exist_ok=True)

# Save test predictions
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_file = os.path.join(output_dir, f'predictions_{timestamp}.json')

predictions_data = {
    'timestamp': timestamp,
    'model_metrics': metrics,
    'test_predictions': [
        {
            'sample': 1,
            'result': result1,
            'description': 'High-risk individual'
        },
        {
            'sample': 2,
            'result': result2,
            'description': 'Low-risk individual'
        }
    ]
}

try:
    with open(results_file, 'w') as f:
        json.dump(predictions_data, f, indent=2)
    print(f"✓ Results saved to Google Drive: {results_file}")
except Exception as e:
    print(f"⚠ Could not save to Drive: {e}")
    print("Make sure Google Drive is mounted above")

print("\n" + "=" * 60)
print("COLAB NOTEBOOK SETUP COMPLETE!")
print("=" * 60)
print("\nYou can now:")
print("1. Use the interactive Gradio interface above to make predictions")
print("2. Modify and test the prediction function with your data")
print("3. Upload your model files from Google Drive")
print("4. Share results back to your Drive automatically")

## How to Use This Notebook

### Quick Start
1. Upload the notebook to Google Colab
2. Run all cells in order from top to bottom
3. Use the interactive Gradio interface to make predictions

### Using Your Own Model
1. Upload the `models` folder from MediPredict to your Google Drive
2. In cell 3, uncomment and update the paths to point to your Drive files
3. Re-run cell 4 to load your model

### Input Features
- **Pregnancies**: Number of times pregnant (0-25)
- **Glucose**: Glucose level in mg/dL (10-500)
- **BloodPressure**: Blood pressure in mmHg (10-300)
- **SkinThickness**: Skin thickness in mm (0-120)
- **Insulin**: Insulin level in uIU/mL (0-2000)
- **BMI**: Body Mass Index (5-100)
- **DiabetesPedigreeFunction**: Genetic predisposition (0-5)
- **Age**: Age in years (1-120)

### Output
The model provides:
- **Prediction**: "Likely Diabetic" or "Likely Non-Diabetic"
- **Confidence**: Probability score (0-100%)
- **Personalized Recommendations**: Health tips based on your inputs

### Share & Save
Results are automatically saved to your Google Drive in the `MediPredict_Results` folder with a timestamp.